### Add this to Add-ons > Install Dependancies & Run to install the packages

In [1]:
# pip install vllm
# pip install cairosvg google_re2
# pip install git+https://github.com/openai/CLIP.git

In [2]:
#| default_exp core

In [3]:
#| export

import torch
import random
import numpy as np
# import multiprocessing as mp
# mp.set_start_method("spawn", force=True)
# Now set your seed
seed = 123
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

import concurrent
import io
import logging
import re
import re2

import cairosvg
import kagglehub
from lxml import etree
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, AutoModel, AutoProcessor
from PIL import Image

from vllm import LLM, SamplingParams
import os

import clip
import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch.nn as nn
from more_itertools import chunked
from PIL import Image, ImageFilter

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

svg_constraints = kagglehub.package_import('metric/svg-constraints')

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print('DEVICE', DEVICE)

def svg_to_png(svg_code: str, size: tuple = (384, 384)) -> Image.Image:
    """
    Converts an SVG string to a PNG image using CairoSVG.

    If the SVG does not define a `viewBox`, it will add one using the provided size.

    Parameters
    ----------
    svg_code : str
        The SVG string to convert.
    size : tuple[int, int], default=(384, 384)
        The desired size of the output PNG image (width, height).

    Returns
    -------
    PIL.Image.Image
        The generated PNG image.
    """
    # Ensure SVG has proper size attributes
    if 'viewBox' not in svg_code:
        svg_code = svg_code.replace('<svg', f'<svg viewBox="0 0 {size[0]} {size[1]}"')

    # Convert SVG to PNG
    png_data = cairosvg.svg2png(bytestring=svg_code.encode('utf-8'))
    return Image.open(io.BytesIO(png_data)).convert('RGB').resize(size)


class ImageProcessor:
    def __init__(self, image: Image.Image, seed=None):
        """Initialize with either a path to an image or a PIL Image object."""
        self.image = image
        self.original_image = self.image.copy()
        if seed is not None:
            self.rng = np.random.RandomState(seed)
        else:
            self.rng = np.random

    def reset(self):
        self.image = self.original_image.copy()
        return self

    def visualize_comparison(
        self,
        original_name='Original',
        processed_name='Processed',
        figsize=(10, 5),
        show=True,
    ):
        """Display original and processed images side by side."""
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=figsize)
        ax1.imshow(np.asarray(self.original_image))
        ax1.set_title(original_name)
        ax1.axis('off')

        ax2.imshow(np.asarray(self.image))
        ax2.set_title(processed_name)
        ax2.axis('off')

        title = f'{original_name} vs {processed_name}'
        fig.suptitle(title)
        fig.tight_layout()
        if show:
            plt.show()
        return fig

    def apply_median_filter(self, size=3):
        """Apply median filter to remove outlier pixel values.

        Args:
            size: Size of the median filter window.
        """
        self.image = self.image.filter(ImageFilter.MedianFilter(size=size))
        return self

    def apply_bilateral_filter(self, d=9, sigma_color=75, sigma_space=75):
        """Apply bilateral filter to smooth while preserving edges.

        Args:
            d: Diameter of each pixel neighborhood
            sigma_color: Filter sigma in the color space
            sigma_space: Filter sigma in the coordinate space
        """
        # Convert PIL Image to numpy array for OpenCV
        img_array = np.asarray(self.image)

        # Apply bilateral filter
        filtered = cv2.bilateralFilter(img_array, d, sigma_color, sigma_space)

        # Convert back to PIL Image
        self.image = Image.fromarray(filtered)
        return self

    def apply_fft_low_pass(self, cutoff_frequency=0.5):
        """Apply low-pass filter in the frequency domain using FFT.

        Args:
            cutoff_frequency: Normalized cutoff frequency (0-1).
                Lower values remove more high frequencies.
        """
        # Convert to numpy array, ensuring float32 for FFT
        img_array = np.array(self.image, dtype=np.float32)

        # Process each color channel separately
        result = np.zeros_like(img_array)
        for i in range(3):  # For RGB channels
            # Apply FFT
            f = np.fft.fft2(img_array[:, :, i])
            fshift = np.fft.fftshift(f)

            # Create a low-pass filter mask
            rows, cols = img_array[:, :, i].shape
            crow, ccol = rows // 2, cols // 2
            mask = np.zeros((rows, cols), np.float32)
            r = int(min(crow, ccol) * cutoff_frequency)
            center = [crow, ccol]
            x, y = np.ogrid[:rows, :cols]
            mask_area = (x - center[0]) ** 2 + (y - center[1]) ** 2 <= r * r
            mask[mask_area] = 1

            # Apply mask and inverse FFT
            fshift_filtered = fshift * mask
            f_ishift = np.fft.ifftshift(fshift_filtered)
            img_back = np.fft.ifft2(f_ishift)
            img_back = np.real(img_back)

            result[:, :, i] = img_back

        # Clip to 0-255 range and convert to uint8 after processing all channels
        result = np.clip(result, 0, 255).astype(np.uint8)

        # Convert back to PIL Image
        self.image = Image.fromarray(result)
        return self

    def apply_jpeg_compression(self, quality=85):
        """Apply JPEG compression.

        Args:
            quality: JPEG quality (0-95). Lower values increase compression.
        """
        buffer = io.BytesIO()
        self.image.save(buffer, format='JPEG', quality=quality)
        buffer.seek(0)
        self.image = Image.open(buffer)
        return self

    def apply_random_crop_resize(self, crop_percent=0.05):
        """Randomly crop and resize back to original dimensions.

        Args:
            crop_percent: Percentage of image to crop (0-0.4).
        """
        width, height = self.image.size
        crop_pixels_w = int(width * crop_percent)
        crop_pixels_h = int(height * crop_percent)

        left = self.rng.randint(0, crop_pixels_w + 1)
        top = self.rng.randint(0, crop_pixels_h + 1)
        right = width - self.rng.randint(0, crop_pixels_w + 1)
        bottom = height - self.rng.randint(0, crop_pixels_h + 1)

        self.image = self.image.crop((left, top, right, bottom))
        self.image = self.image.resize((width, height), Image.BILINEAR)
        return self

    def apply(self):
        """Apply an ensemble of defenses."""
        return (
            self.apply_random_crop_resize(crop_percent=0.03)
            .apply_jpeg_compression(quality=95)
            .apply_median_filter(size=9)
            .apply_fft_low_pass(cutoff_frequency=0.5)
            .apply_bilateral_filter(d=5, sigma_color=75, sigma_space=75)
            .apply_jpeg_compression(quality=92)
        )
    

class AestheticPredictor(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.input_size = input_size
        self.layers = nn.Sequential(
            nn.Linear(self.input_size, 1024),
            nn.Dropout(0.2),
            nn.Linear(1024, 128),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.Dropout(0.1),
            nn.Linear(64, 16),
            nn.Linear(16, 1),
        )

    def forward(self, x):
        return self.layers(x)


class AestheticEvaluator:
    @staticmethod
    def load_models():
        model_path = os.path.join(
            kagglehub.model_download('jiazhuang/sac-logos-ava1-l14-linearmse/Transformers/default/1'),
            'sac+logos+ava1-l14-linearMSE.pth',
        )
        clip_model_path = os.path.join(
            kagglehub.model_download('jiazhuang/clip-vit-large-patch14/Transformers/default/1'),
            'ViT-L-14.pt',
        )
        
        state_dict = torch.load(model_path, weights_only=True, map_location='cuda:1')

        predictor = AestheticPredictor(768)
        predictor.load_state_dict(state_dict)
        predictor.to('cuda:1')
        predictor.eval()

        clip_model, preprocessor = clip.load(clip_model_path, device='cuda:1')

        return predictor, clip_model, preprocessor

    @staticmethod
    def score(image: Image.Image, predictor, clip_model, preprocessor) -> float:
        image = preprocessor(image).unsqueeze(0).to('cuda:1')

        with torch.no_grad():
            image_features = clip_model.encode_image(image)
            image_features /= image_features.norm(dim=-1, keepdim=True)
            image_features = image_features.cpu().detach().numpy()

        score = predictor(torch.from_numpy(image_features).to('cuda:1').float())
        return score.item() / 10.0

    @staticmethod
    def get_score(svg: str) -> float:
        predictor, clip_model, preprocessor = AestheticEvaluator.load_models()

        rng = np.random.RandomState(123)
        group_seed = rng.randint(0, np.iinfo(np.int32).max)

        image_processor = ImageProcessor(image=svg_to_png(svg), seed=group_seed).apply()
        image = image_processor.image.copy()

        return AestheticEvaluator.score(image, predictor, clip_model, preprocessor)
                   



class SVGSanitizer:
    def __init__(self, constraints, default_svg):
        self.constraints = constraints
        self.default_svg = default_svg
    
    def enforce_constraints(self, svg_string: str) -> str:
        """Enforces constraints on an SVG string, removing disallowed elements
        and attributes.

        Parameters
        ----------
        svg_string : str
            The SVG string to process.

        Returns
        -------
        str
            The processed SVG string, or the default SVG if constraints
            cannot be satisfied.
        """
        logging.info('Sanitizing SVG...')

        try:
            parser = etree.XMLParser(remove_blank_text=True, remove_comments=True)
            root = etree.fromstring(svg_string, parser=parser)
        except etree.ParseError as e:
            logging.error('SVG Parse Error: %s. Returning default SVG.', e)
            return self.default_svg
    
        elements_to_remove = []
        for element in root.iter():
            tag_name = etree.QName(element.tag).localname
    
            # Remove disallowed elements
            if tag_name not in self.constraints.allowed_elements:
                elements_to_remove.append(element)
                continue  # Skip attribute checks for removed elements
    
            # Remove disallowed attributes
            attrs_to_remove = []
            for attr in element.attrib:
                attr_name = etree.QName(attr).localname
                if (
                    attr_name
                    not in self.constraints.allowed_elements[tag_name]
                    and attr_name
                    not in self.constraints.allowed_elements['common']
                ):
                    attrs_to_remove.append(attr)
    
            for attr in attrs_to_remove:
                logging.debug(
                    'Attribute "%s" for element "%s" not allowed. Removing.',
                    attr,
                    tag_name,
                )
                del element.attrib[attr]
    
            # Check and remove invalid href attributes
            for attr, value in element.attrib.items():
                 if etree.QName(attr).localname == 'href' and not value.startswith('#'):
                    logging.debug(
                        'Removing invalid href attribute in element "%s".', tag_name
                    )
                    del element.attrib[attr]

            # Validate path elements to help ensure SVG conversion
            if tag_name == 'path':
                d_attribute = element.get('d')
                if not d_attribute:
                    logging.warning('Path element is missing "d" attribute. Removing path.')
                    elements_to_remove.append(element)
                    continue # Skip further checks for this removed element
                # Use regex to validate 'd' attribute format
                path_regex = re2.compile(
                    r'^'  # Start of string
                    r'(?:'  # Non-capturing group for each command + numbers block
                    r'[MmZzLlHhVvCcSsQqTtAa]'  # Valid SVG path commands (adjusted to exclude extra letters)
                    r'\s*'  # Optional whitespace after command
                    r'(?:'  # Non-capturing group for optional numbers
                    r'-?\d+(?:\.\d+)?(?:[Ee][+-]?\d+)?'  # First number
                    r'(?:[\s,]+-?\d+(?:\.\d+)?(?:[Ee][+-]?\d+)?)*'  # Subsequent numbers with mandatory separator(s)
                    r')?'  # Numbers are optional (e.g. for Z command)
                    r'\s*'  # Optional whitespace after numbers/command block
                    r')+'  # One or more command blocks
                    r'\s*'  # Optional trailing whitespace
                    r'$'  # End of string
                )
                if not path_regex.match(d_attribute):
                    logging.warning(
                        'Path element has malformed "d" attribute format. Removing path.'
                    )
                    elements_to_remove.append(element)
                    continue
                logging.debug('Path element "d" attribute validated (regex check).')
        
        # Remove elements marked for removal
        for element in elements_to_remove:
            if element.getparent() is not None:
                element.getparent().remove(element)
                logging.debug('Removed element: %s', element.tag)

        try:
            cleaned_svg_string = etree.tostring(root, encoding='unicode')
            return cleaned_svg_string
        except ValueError as e:
            logging.error(
                'SVG could not be sanitized to meet constraints: %s', e
            )
            return self.default_svg
            

class SVGProcessor:
    @staticmethod
    def clean_and_extract_svgs(text, default_svg):
        text = re.sub(r'^.*?(<svg\b)', r'\1', text, flags=re.DOTALL)
        svg_blocks = re.findall(r'<svg\b.*?</svg>', text, re.DOTALL)
    
        if svg_blocks:
            tmp = re.findall(r'<svg\b.*?', svg_blocks[-1], re.DOTALL)
            if len(tmp) > 1:
                tmp2 = svg_blocks[-1].split('<svg')
                return '<svg ' + tmp2[-1]
            else:
                return svg_blocks[-1]
        else:
            if "<svg" in text and "</svg>" not in text:
                text += "</svg>"
                return text
            return default_svg
    
    @staticmethod
    def svg_conversion_check(topic, base_svg_code, default_svg, sanitizer=None):
        try:
            # Try initial conversion
            cairosvg.svg2png(bytestring=base_svg_code.encode('utf-8'))
            return base_svg_code
        except Exception as e1:
            print(f"Initial conversion failed for '{topic}' due to: {str(e1)}. Attempting with sanitized SVG...")
    
            # if sanitizer:
            #     try:
            #         # Try sanitized conversion
            #         clean_svg_code = sanitizer.enforce_constraints(base_svg_code)
            #         cairosvg.svg2png(bytestring=clean_svg_code.encode('utf-8'))
            #         return clean_svg_code
            #     except Exception as e2:
            #         print(f"Sanitized conversion also failed for '{topic}' due to: {str(e2)}. Returning default SVG.")
            
            return default_svg



class SVGScorer:
    _device = "cuda:1" if torch.cuda.is_available() else "cpu"
    siglip_path=kagglehub.model_download('aishikai/google-siglip-so400m-patch14-384/transformers/default/1')
    _model = AutoModel.from_pretrained(siglip_path).to(_device)
    _processor = AutoProcessor.from_pretrained(siglip_path)

    @staticmethod
    def compute_score(svg,prompt):
        try:
            # Convert SVG to PNG
            temp_png_path = "temp.png"
            cairosvg.svg2png(bytestring=svg.encode('utf-8'), write_to=temp_png_path)
            
            # Open and process the image
            image = Image.open(temp_png_path).convert("RGB")
            texts = ["SVG illustration of " + prompt]
            inputs = SVGScorer._processor(
                text=texts, images=image, padding="max_length", return_tensors="pt"
            ).to(SVGScorer._device)
            
            # Inference without gradient tracking
            with torch.no_grad():
                outputs = SVGScorer._model(**inputs)
            
            logits_per_image = outputs.logits_per_image
            probs = torch.sigmoid(logits_per_image)
            
            return probs[0][0].item()
        
        except Exception as e:
            print(f"An error occurred while scoring SVG: {e}")
            return 0


def add_ocr_decoy_svg(svg_code: str) -> str:
    """
    Adds nested circles with second darkest and second brightest colors from the existing SVG,
    positioned in one of the four corners (randomly selected) but positioned to avoid being
    cropped out during image processing.
    
    Parameters:
    -----------
    svg_code : str
        The original SVG string
    
    Returns:
    --------
    str
        Modified SVG with the nested circles added
    """
    import random
    import re
    from colorsys import rgb_to_hls, hls_to_rgb
    
    # Check if SVG has a closing tag
    if "</svg>" not in svg_code:
        return svg_code
    
    # Extract viewBox if it exists to understand the dimensions
    viewbox_match = re.search(r'viewBox=["\'](.*?)["\']', svg_code)
    if viewbox_match:
        viewbox = viewbox_match.group(1).split()
        try:
            x, y, width, height = map(float, viewbox)
        except ValueError:
            # Default dimensions if we can't parse viewBox
            width, height = 384, 384
    else:
        # Default dimensions if viewBox not found
        width, height = 384, 384
    
    # Function to convert hex color to RGB
    def hex_to_rgb(hex_color):
        hex_color = hex_color.lstrip('#')
        if len(hex_color) == 3:
            hex_color = ''.join([c*2 for c in hex_color])
        return tuple(int(hex_color[i:i+2], 16)/255 for i in (0, 2, 4))
    
    # Function to convert RGB to hex
    def rgb_to_hex(rgb):
        return '#{:02x}{:02x}{:02x}'.format(
            int(rgb[0] * 255), 
            int(rgb[1] * 255), 
            int(rgb[2] * 255)
        )
    
    # Function to calculate color lightness
    def get_lightness(color):
        # Handle different color formats
        if color.startswith('#'):
            rgb = hex_to_rgb(color)
            return rgb_to_hls(*rgb)[1]  # Lightness is the second value in HLS
        elif color.startswith('rgb'):
            rgb_match = re.search(r'rgb\((\d+),\s*(\d+),\s*(\d+)\)', color)
            if rgb_match:
                r, g, b = map(lambda x: int(x)/255, rgb_match.groups())
                return rgb_to_hls(r, g, b)[1]
        return 0.5  # Default lightness if we can't parse
    
    # Extract all colors from the SVG
    color_matches = re.findall(r'(?:fill|stroke)="(#[0-9A-Fa-f]{3,6}|rgb\(\d+,\s*\d+,\s*\d+\))"', svg_code)
    
    # Default colors in case we don't find enough
    second_darkest_color = "#333333"  # Default to dark gray
    second_brightest_color = "#CCCCCC"  # Default to light gray
    
    if color_matches:
        # Remove duplicates and get unique colors
        unique_colors = list(set(color_matches))
        
        # Calculate lightness for each unique color
        colors_with_lightness = [(color, get_lightness(color)) for color in unique_colors]
        
        # Sort by lightness (brightness)
        sorted_colors = sorted(colors_with_lightness, key=lambda x: x[1])
        
        # Handle different scenarios based on number of unique colors
        if len(sorted_colors) >= 4:
            # We have at least 4 unique colors - use 2nd darkest and 2nd brightest
            second_darkest_color = sorted_colors[1][0]
            second_brightest_color = sorted_colors[-2][0]
        elif len(sorted_colors) == 3:
            # We have 3 unique colors - use 2nd darkest and brightest
            second_darkest_color = sorted_colors[1][0]
            second_brightest_color = sorted_colors[2][0]
        elif len(sorted_colors) == 2:
            # We have only 2 unique colors - use the darkest and brightest
            second_darkest_color = sorted_colors[0][0]
            second_brightest_color = sorted_colors[1][0]
        elif len(sorted_colors) == 1:
            # Only one color - use it for second_darkest and a derived lighter version
            base_color = sorted_colors[0][0]
            base_lightness = sorted_colors[0][1]
            second_darkest_color = base_color
            
            # Create a lighter color variant if the base is dark, or darker if base is light
            if base_lightness < 0.5:
                # Base is dark, create lighter variant
                second_brightest_color = "#CCCCCC"
            else:
                # Base is light, create darker variant
                second_darkest_color = "#333333"
    
    # Ensure the colors are different
    if second_darkest_color == second_brightest_color:
        # If they ended up the same, modify one of them
        if get_lightness(second_darkest_color) < 0.5:
            # It's a dark color, make the bright one lighter
            second_brightest_color = "#CCCCCC"
        else:
            # It's a light color, make the dark one darker
            second_darkest_color = "#333333"
    
    # Base size for the outer circle
    base_outer_radius = width * 0.023
    
    # Randomize size by ±10%
    size_variation = base_outer_radius * 0.1
    outer_radius = base_outer_radius + random.uniform(-size_variation, size_variation)
    
    # Define radii for inner circles based on outer radius
    middle_radius = outer_radius * 0.80
    inner_radius = middle_radius * 0.65
    
    # Calculate the maximum crop margin based on the image processing (5% of dimensions)
    # Add 20% extra margin for safety
    crop_margin_w = int(width * 0.05 * 1.2)
    crop_margin_h = int(height * 0.05 * 1.2)
    
    # Calculate center point based on the outer radius to ensure the entire circle stays visible
    safe_offset = outer_radius + max(crop_margin_w, crop_margin_h)
    
    # Choose a random corner (0: top-left, 1: top-right, 2: bottom-left, 3: bottom-right)
    corner = random.randint(0, 3)
    
    # Position the circle in the chosen corner, accounting for crop margin
    if corner == 0:  # Top-left
        center_x = safe_offset
        center_y = safe_offset
    elif corner == 1:  # Top-right
        center_x = width - safe_offset
        center_y = safe_offset
    elif corner == 2:  # Bottom-left
        center_x = safe_offset
        center_y = height - safe_offset
    else:  # Bottom-right
        center_x = width - safe_offset
        center_y = height - safe_offset
    
    # Add a small random offset (±10% of safe_offset) to make positioning less predictable
    random_offset = safe_offset * 0.1
    center_x += random.uniform(-random_offset, random_offset)
    center_y += random.uniform(-random_offset, random_offset)
    
    # Round to 1 decimal place to keep file size down
    outer_radius = round(outer_radius, 1)
    middle_radius = round(middle_radius, 1)
    inner_radius = round(inner_radius, 1)
    center_x = round(center_x, 1)
    center_y = round(center_y, 1)
    
    # Create the nested circles
    outer_circle = f'<circle cx="{center_x}" cy="{center_y}" r="{outer_radius}" fill="{second_darkest_color}" />'
    middle_circle = f'<circle cx="{center_x}" cy="{center_y}" r="{middle_radius}" fill="{second_brightest_color}" />'
    inner_circle = f'<circle cx="{center_x}" cy="{center_y}" r="{inner_radius}" fill="{second_darkest_color}" />'
    
    # Create a group element that contains all three circles
    group_element = f'<g>{outer_circle}{middle_circle}{inner_circle}</g>'
    
    # Insert the group element just before the closing SVG tag
    modified_svg = svg_code.replace("</svg>", f"{group_element}</svg>")
    
    # Calculate and add a comment with the byte size information
    outer_bytes = len(outer_circle.encode('utf-8'))
    middle_bytes = len(middle_circle.encode('utf-8'))
    inner_bytes = len(inner_circle.encode('utf-8'))
    total_bytes = outer_bytes + middle_bytes + inner_bytes
    
    corner_names = ["top-left", "top-right", "bottom-left", "bottom-right"]
    byte_info = f'<!-- Circle bytes: outer={outer_bytes}, middle={middle_bytes}, ' \
                f'inner={inner_bytes}, total={total_bytes}, ' \
                f'colors: dark={second_darkest_color}, light={second_brightest_color}, ' \
                f'position: {corner_names[corner]} -->'
    
    modified_svg = modified_svg.replace("</svg>", f"{byte_info}</svg>")
    
    return modified_svg

######################----------------------------------
#### high score svg
import xml.etree.ElementTree as ET
import re

def high_score_svg_resize(
    svg_code: str,
    padding_ratio: float = 0.08,  # Padding for the border area relative to original size
    min_stroke: float = 1.5,
    max_stroke: float = 16,
) -> str:
    """
    Enhances the SVG code by adjusting the stroke width, font size, scaling it,
    and adding a solid black border (padding area), preserving the original aspect ratio.

    Parameters:
    svg_code (str): The original SVG code.
    padding_ratio (float): Padding to apply around the SVG (default 0.08). This is the area
                           reserved for the solid black border, relative to the original largest dimension.
    min_stroke (float): Minimum stroke width (default 1.5).
    max_stroke (float): Maximum stroke width (default 16).

    Returns:
    str: The enhanced SVG code with a solid black border, preserving the original aspect ratio.
    """
    root = ET.fromstring(svg_code)

    # Define structural tags early
    structural_tags = {
        ET.QName("http://www.w3.org/2000/svg", ln)
        for ln in ['defs', 'style', 'title', 'metadata', 'script']
    }

    viewBox = root.get("viewBox")
    if viewBox is None:
        # Attempt to infer viewBox from width/height if available
        width = root.get("width")
        height = root.get("height")
        if width and height:
            try:
                vb_w = float(re.findall(r'\d+\.?\d*', width)[0])
                vb_h = float(re.findall(r'\d+\.?\d*', height)[0])
                viewBox = f"0 0 {vb_w} {vb_h}"
                root.set("viewBox", viewBox)
            except (ValueError, IndexError):
                viewBox = "0 0 96 96"  # Default if parsing fails
                root.set("viewBox", viewBox)
        else:
            viewBox = "0 0 96 96"  # Default if no viewBox, width, or height
            root.set("viewBox", viewBox)

    vb_x, vb_y, vb_w, vb_h = map(float, viewBox.strip().split())

    # Determine the reference size for proportional calculations (the larger dimension)
    original_ref_size = max(vb_w, vb_h)
    if original_ref_size <= 0: # Handle case where viewBox is invalid
        original_ref_size = 100 # Default reference size

    # Calculate padding relative to the original largest dimension
    total_padding_on_one_side = padding_ratio * original_ref_size

    # Calculate the new canvas dimensions including the padding
    new_canvas_width = vb_w + 2 * total_padding_on_one_side
    new_canvas_height = vb_h + 2 * total_padding_on_one_side

    # Calculate translation to position the original content within the padded area
    content_translate_x = total_padding_on_one_side - vb_x
    content_translate_y = total_padding_on_one_side - vb_y

    # Create a group for the original SVG content
    g_inner = ET.Element("g")
    transform_inner = f"translate({content_translate_x:.2f},{content_translate_y:.2f})"
    g_inner.set("transform", transform_inner)

    # Remove structural elements from the main root and move content to inner group
    children_to_move = []
    for child in list(root):
        qname = ET.QName(child.tag)
        if qname not in structural_tags:
            children_to_move.append(child)

    for child in children_to_move:
        g_inner.append(child)
        root.remove(child)

    # Create the new root element with the updated dimensions and viewBox
    new_root = ET.Element("svg", xmlns="http://www.w3.org/2000/svg")
    new_root.set("width", str(round(new_canvas_width)))
    new_root.set("height", str(round(new_canvas_height)))
    new_root.set("viewBox", f"0 0 {round(new_canvas_width)} {round(new_canvas_height)}")
    # Preserve aspect ratio is now handled by the calculated new canvas dimensions and viewBox

    # --- Add Solid Black Border ---
    black_border_rect = ET.Element("rect")
    black_border_rect.set("x", "0")
    black_border_rect.set("y", "0")
    black_border_rect.set("width", str(round(new_canvas_width)))
    black_border_rect.set("height", str(round(new_canvas_height)))
    black_border_rect.set("fill", "#000000") # Fill the entire new canvas with black
    black_border_rect.set("stroke", "none")
    new_root.append(black_border_rect)

    # Add a white rectangle behind the inner content to make it opaque and cover the black
    # in the content area. The original content will be placed on top of this.
    white_background_rect = ET.Element("rect")
    white_background_rect.set("x", str(total_padding_on_one_side))
    white_background_rect.set("y", str(total_padding_on_one_side))
    white_background_rect.set("width", str(round(vb_w))) # Use original content dimensions
    white_background_rect.set("height", str(round(vb_h))) # Use original content dimensions
    white_background_rect.set("fill", "#FFFFFF") # Fill with white
    white_background_rect.set("stroke", "none")
    new_root.append(white_background_rect)


    # Append the transformed inner content group
    new_root.append(g_inner)

    # Adjust stroke-width and font-size to fit the new relative scale
    def clamp_visuals(el):
        for attr in ("stroke-width", "font-size"):
            if attr in el.attrib:
                try:
                    original = float(re.findall(r'\d+\.?\d*', el.attrib[attr])[0])
                    clamped = max(min_stroke, min(max_stroke, original))
                    el.attrib[attr] = f"{clamped:.2f}"
                except (ValueError, IndexError):
                    pass
        for child in el:
            clamp_visuals(child)

    # Apply clamping to the inner content group
    clamp_visuals(g_inner)

    # Return the XML as a string
    return ET.tostring(new_root, encoding="unicode")

################----------------------------------------------------------------

class Model:
    def __init__(self):
        self.model_path = kagglehub.model_download('vinothkumarsekar89/llama_3p2_3b_svg_16bit_merged/transformers/04') 
        
        self.llm = LLM(
            model=self.model_path,
            max_model_len=1024,
            #tensor_parallel_size=2,
            gpu_memory_utilization=0.85,
            #trust_remote_code=True,
            dtype="half",
            #enforce_eager=True,
            seed=123,
            disable_log_stats=True
        )
        
        self.default_svg = """<svg width="256" height="256" viewBox="0 0 256 256"><circle cx="50" cy="50" r="40" fill="red" /></svg>"""
        self.constraints = svg_constraints.SVGConstraints()
        self.sanitizer = SVGSanitizer(self.constraints, self.default_svg)
        self.N=5
        
    def get_response(self, description_list):
        alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.
    
        ### Instruction:
        Generate a SVG code for the given input:
    
        ### Input:
        {}
    
        ### Response:
        """
        formatted_input_list = [alpaca_prompt.format(description) for description in description_list]
        sampling_params = SamplingParams(temperature=0.7, top_k=40, top_p=0.95, max_tokens=1024)
        outputs = self.llm.generate(formatted_input_list, sampling_params)
        
        #suitable for batch inputs as well
        output_list=[]
        for output in outputs:
            generated_text = output.outputs[0].text
            output_list.append(generated_text.strip())
    
        return output_list
        
    def process_svg(self, description_list: list) -> list:  
        # Get the response and clean, extract, and check SVG code
    
        output_decoded_list = self.get_response(description_list)  
        clean_svg_code_list = []  
      
        # Iterate over the output and corresponding description  
        for output_decoded, description in zip(output_decoded_list, description_list):  
            # Clean and extract SVG code  
            base_svg_code = SVGProcessor.clean_and_extract_svgs(output_decoded, self.default_svg)  
              
            # Enforce constraints on the SVG code  
            clean_svg_code = self.sanitizer.enforce_constraints(base_svg_code)  
              
            # Check and convert the SVG code  
            clean_svg_code = SVGProcessor.svg_conversion_check(description, base_svg_code, self.default_svg)  
                               
            # Add the cleaned SVG code to the list  
            clean_svg_code_list.append(clean_svg_code)  
      
        return clean_svg_code_list  
        
    
    def predict(self, description: str, max_new_tokens=1024) -> str:
        
        # Process  SVGs
        try:
            clean_svg_code_list = self.process_svg([description]*self.N)
        except Exception as e:
            logging.error(f"Error while process SVGs: {e}")
            return self.default_svg
            
        max_score = 0.0  
        best_svg = self.default_svg  

        try:
            # Iterate over the SVGs and their descriptions  
            for svg, description in zip(clean_svg_code_list, [description]*self.N):  
                # Compute the score for the current SVG and description
                               
                score = SVGScorer.compute_score(svg, description)
                #print('score',score)
                aes_score=AestheticEvaluator.get_score(svg)
                #print('aes score',aes_score)
                
                combined_score= ((2*score)+(aes_score))/3
                if (combined_score > 0.6):
                    return svg
                else:
                    if combined_score > max_score:  
                        max_score = combined_score
                        best_aes_score= aes_score
                        best_svg = svg
        
                logging.info(f"Current max score: {max_score}")
                
            return best_svg
            
        except Exception as e:
            logging.error(f"Error while SL-scoring SVGs: {e}")
            return self.default_svg

2025-05-14 15:40:26.418378: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1747237226.807778      44 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1747237226.941588      44 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


INFO 05-14 15:40:43 [importing.py:53] Triton module has been replaced with a placeholder.
INFO 05-14 15:40:43 [__init__.py:239] Automatically detected platform cuda.
DEVICE cuda


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


In [4]:
# model=Model()

# import time

# start_time = time.time()

# # Replace this with your actual model call
# model.predict('sun rising in the east')

# end_time = time.time()
# elapsed_time = end_time - start_time

# #target secs/sample: 50-55s max
# print(f"Prediction took {elapsed_time:.4f} seconds for single sample")
# print(f"Prediction took {elapsed_time*500/3600:.4f} Hrs for 500 samples")

In [5]:
# import time

# start_time = time.time()

# # Replace this with your actual model call
# model.predict('sun rising in the east')

# end_time = time.time()
# elapsed_time = end_time - start_time

# #target secs/sample: 50-55s max
# print(f"Prediction took {elapsed_time:.4f} seconds for single sample")
# print(f"Prediction took {elapsed_time*500/3600:.4f} Hrs for 500 samples")

In [6]:
import sys
#sys.path.append('/kaggle/input/drawing-with-llms/published/')
import kaggle_evaluation

logging.basicConfig(level=logging.INFO, force=True)
kaggle_evaluation.test(Model)
print('Run success!')

Creating Model instance...
WARNING 05-14 15:41:30 [config.py:2972] Casting torch.bfloat16 to torch.float16.
INFO 05-14 15:41:43 [config.py:717] This model supports multiple tasks: {'reward', 'generate', 'score', 'classify', 'embed'}. Defaulting to 'generate'.
WARNING 05-14 15:41:43 [arg_utils.py:1658] Compute Capability < 8.0 is not supported by the V1 Engine. Falling back to V0. 
INFO 05-14 15:41:43 [llm_engine.py:240] Initializing a V0 LLM engine (v0.8.5.post1) with config: model='/kaggle/input/llama_3p2_3b_svg_16bit_merged/transformers/04/1', speculative_config=None, tokenizer='/kaggle/input/llama_3p2_3b_svg_16bit_merged/transformers/04/1', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=1024, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=Fal

[W514 15:41:56.756160827 socket.cpp:204] [c10d] The hostname of the client socket cannot be retrieved. err=-3


INFO 05-14 15:42:06 [parallel_state.py:1004] rank 0 in world size 1 is assigned as DP rank 0, PP rank 0, TP rank 0
INFO 05-14 15:42:06 [model_runner.py:1108] Starting to load model /kaggle/input/llama_3p2_3b_svg_16bit_merged/transformers/04/1...


[W514 15:42:06.766742766 socket.cpp:204] [c10d] The hostname of the client socket cannot be retrieved. err=-3


Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]


INFO 05-14 15:42:41 [loader.py:458] Loading weights took 35.02 seconds
INFO 05-14 15:42:41 [model_runner.py:1140] Model loading took 6.0156 GiB and 35.210608 seconds
INFO 05-14 15:42:44 [worker.py:287] Memory profiling takes 1.53 seconds
INFO 05-14 15:42:44 [worker.py:287] the current vLLM instance can use total_gpu_memory (14.74GiB) x gpu_memory_utilization (0.85) = 12.53GiB
INFO 05-14 15:42:44 [worker.py:287] model weights take 6.02GiB; non_torch_memory takes 0.05GiB; PyTorch activation peak memory takes 1.18GiB; the rest of the memory reserved for KV Cache is 5.29GiB.
INFO 05-14 15:42:44 [executor_base.py:112] # cuda blocks: 3093, # CPU blocks: 2340
INFO 05-14 15:42:44 [executor_base.py:117] Maximum concurrency for 1024 tokens per request: 48.33x
INFO 05-14 15:42:50 [model_runner.py:1450] Capturing cudagraphs for decoding. This may lead to unexpected consequences if the model is not static. To run the model in eager mode, set 'enforce_eager=True' or use '--enforce-eager' in the CLI.

Capturing CUDA graph shapes:   0%|          | 0/35 [00:00<?, ?it/s]

INFO 05-14 15:43:27 [model_runner.py:1592] Graph capturing finished in 38 secs, took 0.19 GiB
INFO 05-14 15:43:27 [llm_engine.py:437] init engine (profile, create kv cache, warmup model) took 45.70 seconds
Running inference tests...
Wrote test submission file to "/tmp/kaggle-evaluation-submission-3622n_mn.csv".
Success!
Run success!
